## Let's run intron clustering to annotate alternative splicing events given observed junctions in our cells 

In [77]:
# Load LeafletSC 
import LeafletSC
import os
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns

# turn this into AnnData object 
import anndata as ad
from scipy.sparse import csr_matrix
from scipy.sparse import coo_matrix

from LeafletSC.clustering.find_intron_clusters import main as find_intron_clusters
from LeafletSC.clustering.prepare_model_input import main as prep_model_input
from LeafletSC.clustering.find_intron_clusters import visualize_local_events

# Define path that contains junction files
juncs_path = "/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaSenis/junctions/"
print("The junctions are loaded from the following path: " + juncs_path) 

# print the files in the path 
print("The files in the path are: " + str(os.listdir(juncs_path)))

# define path for saving the output data 
output_path = "/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaSenis/Leaflet"

# we provide a gtf file for the human genome as well to make better sense of the junctions that are detected in cells
# please replace with the path to the gtf file on your system
gtf_file="/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaSenis/genome_files/gencode.vM19/genes/genes.gtf" 

The junctions are loaded from the following path: /gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaSenis/junctions/
The files in the path are: ['3_month', '24_month', '21_month', '18_month']


In [78]:
!hostname

pe2cc2-075.c.nygenome.org


In [102]:
# metadata file
metadata = "/gpfs/commons/projects/knowles_singlecell_splicing/TabulaSenis/data/AWS/metadata/tabula-muris-senis-full-metadata.csv"
metadata = pd.read_csv(metadata)
metadata.head()

/scratch/ipykernel_7919/2948469132.py:3: DtypeWarning: Columns (3,5,14) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv(metadata)


,index,age,batch,cell,cell_ontology_class,cell_ontology_id,free_annotation,method,mouse.id,n_counts,n_genes,sex,subtissue,tissue,tissue_free_annotation,louvain,leiden
0,AAACCTGCAGGGTACA-1-0-0-0-0,24m,0,MACA_24m_M_TONGUE_60_AAACCTGCAGGGTACA,keratinocyte,NaN,filiform,droplet,24-M-60,5482.0,2107.0,male,NaN,Tongue,Tongue,5,3
1,AAACCTGCAGTAAGCG-1-0-0-0-0,24m,0,MACA_24m_M_TONGUE_60_AAACCTGCAGTAAGCG,keratinocyte,NaN,suprabasal,droplet,24-M-60,21855.0,3481.0,male,NaN,Tongue,Tongue,27,31
2,AAACCTGTCATTATCC-1-0-0-0-0,24m,0,MACA_24m_M_TONGUE_60_AAACCTGTCATTATCC,keratinocyte,NaN,suprabasal,droplet,24-M-60,10942.0,2599.0,male,NaN,Tongue,Tongue,27,31
3,AAACGGGGTACAGTGG-1-0-0-0-0,24m,0,MACA_24m_M_TONGUE_60_AAACGGGGTACAGTGG,keratinocyte,NaN,suprabasal differentiating,droplet,24-M-60,20665.0,3468.0,male,NaN,Tongue,Tongue,24,11
4,AAACGGGGTCTTCTCG-1-0-0-0-0,24m,0,MACA_24m_M_TONGUE_60_AAACGGGGTCTTCTCG,keratinocyte,NaN,suprabasal differentiating,droplet,24-M-60,12925.0,3189.0,male,NaN,Tongue,Tongue,5,3


In [103]:
# subset metadata to only facs under method 
metadata = metadata[metadata['method'] == 'facs']

In [104]:
metadata.age.value_counts()

age
3m     44518
18m    34027
24m    31551
21m      728
Name: count, dtype: int64

### Let's first define some parameters for the analysis

In [105]:
juncs_path

'/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaSenis/junctions/'

In [106]:
# Modify cell IDs based on the month group
def format_cell_id(index, group):
    if group == '3m':
        # Modify for 3-month group: replace first '.' and '_', and fix '3-39' to '3_39'
        parts = index.replace('.', '-', 1).replace('_', '-', 1).split('.')
        # Replace hyphen in '3-39' to '3_39'
        corrected_part = parts[1].replace('-', '_', 1)
        # Append '-1-1' and return
        return parts[0] + '-' + corrected_part + '-1-1'
    else:
        # For other groups, keep the standard format and retain everything including the last '-1-1'
        return index.split('.')[0]

# Apply the function based on the 'age' column in your metadata
metadata['cell_id'] = metadata.apply(lambda row: format_cell_id(row['index'], row['age']), axis=1)

cell_ids = metadata['cell_id'].values
cell_ids_set = set(cell_ids)

In [108]:
all_junc_files = []

# List all subdirectories in juncs_path
all_dirs = os.listdir(juncs_path)
print("Subdirectories found:", all_dirs)

# Loop through each subdirectory
for dir in all_dirs:
    dir_path = os.path.join(juncs_path, dir)
    # Check if it's actually a directory
    if os.path.isdir(dir_path):
        print(dir_path)
        junc_files = os.listdir(dir_path)
        print(len(junc_files))
        
        # Construct full paths and include the subdirectory information
        junc_files = [os.path.join(dir_path, x) for x in junc_files]
        
        # Add these files to the main list
        all_junc_files.extend(junc_files)

Subdirectories found: ['3_month', '24_month', '21_month', '18_month']
/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaSenis/junctions/3_month
50650
/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaSenis/junctions/24_month
55362
/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaSenis/junctions/21_month
1757
/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaSenis/junctions/18_month
53549


In [110]:
# Filter the files using the set for faster lookup and keep the correct paths
portion_in_list2 = [x for x in all_junc_files if os.path.splitext(os.path.basename(x))[0] in cell_ids_set]
len(portion_in_list2)

107574

In [111]:
portion_not_in_list2 = [x for x in all_junc_files if os.path.splitext(os.path.basename(x))[0] not in cell_ids_set]

In [ ]:
# Construct final paths by appending "/junctions_with_barcodes.bed" to each matching file
portion_in_list2 = [x + "/junctions_with_barcodes.bed" for x in portion_in_list2]

In [ ]:
# junc_files defines a path for where junction files can be found, in this case, the path is defined above
junc_files = portion_in_list2
print(len(junc_files), len(cell_ids))

In [ ]:
# define additional parameters 
sequencing_type = "single_cell"

# ensure output files are to be saved in output_path 
output_file = output_path + "/tabula_senis_all_intron_clusters"
junc_bed_file= output_path + "/tabula_senis_all_intron_clusters.bed" # you can load this file into IGV to visualize the junction coordinates 
min_intron_length = 50
max_intron_length = 500000
threshold_inc = 0.01 
min_junc_reads = 5
min_num_cells_wjunc = 5
keep_singletons = False # ignore junctions that do not share splice sites with any other junction (likely const)
junc_suffix = "*with_barcodes.bed" # depends on how you ran regtools 

### Run intron clustering 

In [ ]:
all_juncs_df = find_intron_clusters(junc_files=junc_files, gtf_file=gtf_file, output_file=output_file, 
                       sequencing_type=sequencing_type, junc_bed_file=junc_bed_file, 
                       threshold_inc=threshold_inc, min_intron = min_intron_length,
                       max_intron=max_intron_length, min_junc_reads=min_junc_reads,
                       singleton=keep_singletons,
                       junc_suffix=junc_suffix, min_num_cells_wjunc=min_num_cells_wjunc, filter_shared_ss=True, 
                       run_notebook = True)

In [ ]:
# load bed file 
import pandas as pd
juncs_bed = pd.read_csv(junc_bed_file, sep="\t", header=None)
# remove columns 3 and 4 
juncs_bed = juncs_bed.drop(columns=[3, 4])
juncs_bed.columns = ["Chromosome", "Start", "End", "Strand", "junction_id", "Start_b", "End_b", "gene_id", "gene_name", "transcript_id", "exon_id"]

# cmobine with all_juncs_df and prep df for visualization 
dat_vis = all_juncs_df[["chrom", "chromStart", "chromEnd", "strand", "intron_length", "counts_total", "junction_id", "Cluster"]]
dat_vis = dat_vis.drop_duplicates()
# merge dat_vis with juncs_bed using all common columns 
dat_vis = dat_vis.merge(juncs_bed, how="left", on=["junction_id"])

### Stop here for now

### Now let's convert the intron clusters to a format that can be used by LeafletSC

In [ ]:
intron_clusters = "" # path to the intron clusters file
output_file = output_path + "Leaflet_ALL_months__model_input" # name of the output file
has_genes = "yes" # since we used a gtf file to obtain the intron clusters, we can set this to yes
chunk_size = 5000 # number of junctions to process at a time from the intron clusters files
# metadata = None # can replace with path, if metadata is available for cells (cell type, origin, library ID...)

In [ ]:
prep_model_input(intron_clusters, output_file, has_genes, chunk_size, metadata)

### Take a quick look at the input file that will go into the model to get familiarized with all the columns

In [ ]:
model_input_data = ""
summarized_data = pd.read_hdf(model_input_data, 'df')
print(summarized_data.head())

#### Note that fow now, the values in cell_type default to the cell's path, in the future it will be possible to specify the cell type in the metadata file

In [ ]:
# let's see all the columns in the summarized data
print(summarized_data.columns)

#### We can quickly visualize the overall junction usage ratio distribution across all cells

In [ ]:
summarized_data.head()

In [ ]:
print(len(summarized_data.cell_id.unique()))

In [ ]:
all_cell_ids = summarized_data.cell_id

# clean up each cell_id using .str.split("_/gpfs")[0]
all_cell_ids_clean = [x.split("_/gpfs")[0] for x in all_cell_ids]
summarized_data["cell_id_clean"] = all_cell_ids_clean

In [ ]:
summarized_data.head()

In [ ]:
# remove cell_id columb from summarized_data 
summarized_data = summarized_data.drop(columns=["cell_id"])
# rename cell_id_clean to cell_id
summarized_data = summarized_data.rename(columns={"cell_id_clean": "cell_id"})
summarized_data.head()

In [ ]:
summarized_data.drop(columns="cell_type", inplace=True)

In [ ]:
metadata = metadata[["cell_id", "age", "batch", "cell_ontology_class", "free_annotation", "mouse.id", "sex", "subtissue", "tissue", "tissue_free_annotation", "louvain", "leiden"]]
# merge summarized_data with metadata
summarized_data = summarized_data.merge(metadata, on="cell_id")

In [ ]:
# ensure cell_type column is present 
summarized_data = summarized_data.rename(columns={"cell_ontology_class": "cell_type"})
summarized_data.head()

In [ ]:
cell_vars = summarized_data[['cell_id_index', 'cell_id', 'age', 'batch', "cell_type", "mouse.id", "sex", "subtissue", "tissue", "louvain", "leiden"]].drop_duplicates()
cell_vars.head()

In [ ]:
# Specify the junction variables
junction_vars = summarized_data[['Cluster', 'junction_id', 'gene_id', 'junction_id_index']].drop_duplicates()
junction_vars.head()

In [ ]:
# Create sparse matrices directly using cell_id_index, junction_id_index, and counts
n_cells = cell_vars['cell_id_index'].nunique()
n_junctions = junction_vars['junction_id_index'].nunique()

In [ ]:
test=summarized_data[["cell_id_index", "Cluster_Counts", "Cluster"]].merge(junction_vars, on=['Cluster'], how='right').drop_duplicates()
test.head()

In [ ]:
# Sparse matrix for junc_count
junc_count_sparse = coo_matrix(
    (summarized_data['junc_count'], 
     (summarized_data['cell_id_index'], summarized_data['junction_id_index'])),
    shape=(n_cells, n_junctions)
)

# Sparse matrix for Cluster_Counts 
# Here cell-junction pairs that are zero while cell-cluster (to which junction belongs to is not zero) 
# Those indices have to have cluster counts in this matrix 
cluster_count_sparse = coo_matrix(
    (test['Cluster_Counts'], 
     (test['cell_id_index'], test['junction_id_index'])),
    shape=(n_cells, n_junctions)
)

In [ ]:
# Create an AnnData object with junc_count_sparse
adata = ad.AnnData(X=junc_count_sparse, obs=cell_vars.set_index('cell_id_index'), var=junction_vars.set_index('junction_id_index'))

# Add the second sparse matrix to the AnnData object as a new layer
adata.layers["Junction_Counts"] = junc_count_sparse
adata.layers["Cluster_Counts"] = cluster_count_sparse

adata.var.index = adata.var.index.astype(int)
adata.var.sort_index(ascending=True, inplace=True)

adata.obs.index = adata.obs.index.astype(int)
adata.obs.sort_index(ascending=True, inplace=True)

# Convert the sparse matrix in adata.X to CSR format
adata.X = csr_matrix(adata.X)

# If you have other sparse matrices in layers, convert them as well
adata.layers["Junction_Counts"] = csr_matrix(adata.layers["Junction_Counts"])
# If you have other sparse matrices in layers, convert them as well
adata.layers["Cluster_Counts"] = csr_matrix(adata.layers["Cluster_Counts"])

In [ ]:
# Extract junction counts and cluster counts
junction_counts = adata.layers["Junction_Counts"].toarray()  # Convert sparse to dense if necessary
cluster_counts = adata.layers["Cluster_Counts"].toarray()

adata.layers["junc_ratio"] = np.divide(junction_counts, cluster_counts, out=np.zeros_like(junction_counts, dtype=float), where=(cluster_counts != 0))


In [ ]:
# Save the AnnData object to a file in HDF5 format
adata.write_h5ad('/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaSenis/Leaflet_ALL_months_ALL_tissues_intron_clusters_adata.h5ad', compression='gzip')